# Task 10: the low-fidelity drone grid on Colab CPUs

Runs the 9^6 design grid of `docs/specs/task10_drone_sizing.md` (section 8) through the low-fidelity
model with every core of the runtime, and writes the grid, its exact Pareto front and a timing record
to Drive. **Requires T10.5 and T10.6 merged** (`drone/evaluate.py`, `physics/gates.py` family `drone`,
`analysis/drone_grid.py`); until then the import cell below fails on purpose.

Every setup cell skips work already done. A CPU runtime with 8 vCPUs is enough; the notebook prints
the measured cost per design before it commits to the whole grid.

## 0. Environment

In [ ]:
!nproc
!free -h
import os
N_WORKERS = os.cpu_count()

## 1. Repository
Clones the repo at `COMMIT`, installs it, prints the sha that every record of this run will carry.
Restart the runtime after changing `COMMIT`.

In [ ]:
COMMIT = "main"   # branch, tag or commit sha; pin it to the commit that holds the task named above
import os, subprocess, sys
if not os.path.isdir("/content/ai-engineering"):
    !git clone -q https://github.com/emarinhoss/ai-engineering.git /content/ai-engineering
%cd /content/ai-engineering
stale = [m for m in sys.modules if m.split(".")[0] in ("physics", "geometry", "drone", "surrogate", "analysis", "designs")]
assert not stale, f"repo modules already imported in this kernel ({stale[:3]}...): Runtime -> Restart runtime, then run again"
!git fetch -q --all
!git checkout -q {COMMIT}
!git pull -q --ff-only
!pip install -q -e .
sys.path.insert(0, "/content/ai-engineering")
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", HEAD)

## 2. Google Drive output folder
Records land under `su2queue/drone_grid/<stamp>_<sha7>/` on Drive, one folder per run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = "/content/drive/MyDrive/su2queue/drone_grid"
os.makedirs(OUT_ROOT, exist_ok=True)
print("output root:", OUT_ROOT)

## 3. Cost of one design
Evaluates the nominal design once (cold) and ten random designs (warm) and prints the seconds per
design. Look at: the warm figure times 531 441 / `N_WORKERS` is the grid's wall time. The spec expects
under a second per design; if it is far above that, stop here and report.

In [ ]:
import time, numpy as np
from drone import evaluate, ledger
from designs.make_pool_drone import INPUTS, LOW, HIGH
fixed = ledger.load_all()
nominal = {"family": "drone", "fidelity": "low", "tag": "grid", "span_m": 2.0, "aspect_ratio": 10.0,
           "taper_ratio": 0.6, "battery_fraction": 0.30, "camber_pct": 4.0, "thickness_pct": 12.0}
t0 = time.perf_counter(); out = evaluate.evaluate(nominal, fixed); cold = time.perf_counter() - t0
rng = np.random.default_rng(0)
t0 = time.perf_counter()
for _ in range(10):
    x = LOW + rng.random(len(INPUTS)) * (np.array(HIGH) - np.array(LOW))
    evaluate.evaluate({**nominal, **dict(zip(INPUTS, map(float, x)))}, fixed)
warm = (time.perf_counter() - t0) / 10
print(f"nominal: endurance {out['endurance_h']:.3f} h, mtow {out['mtow_kg']:.3f} kg, gates {out['gates']}")
print(f"cold {cold:.3f} s, warm {warm:.3f} s per design; grid estimate {warm * 9**6 / N_WORKERS / 3600:.2f} h on {N_WORKERS} workers")

## 4. The grid (this cell runs for the estimated time)
`analysis.drone_grid.run` evaluates every gate-passing point of the 9^6 grid in a multiprocessing pool,
writes `grid.csv` (one row per design with every output and the gate messages, failed closures included
with their error) and `front.csv` (the exact Pareto front of endurance against take-off mass), plus
`record.json` with the commit, the ledger hashes, the worker count and the wall time. `--points 9` is
the spec's grid; use `--points 5` first for a 15 625-point rehearsal.

In [ ]:
import datetime, subprocess
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + HEAD[:7]
OUT = f"{OUT_ROOT}/{STAMP}"
POINTS = 9        # 5 for a rehearsal
!python -m analysis.drone_grid --points {POINTS} --workers {N_WORKERS} --out {OUT}
print(open(f"{OUT}/record.json").read())

## 5. Look at the result
Prints the size of the front and the ranges of both objectives, and draws the exact front. Look at: the
front should be a monotone curve (more endurance costs more mass) and no design on it may carry a gate
message.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
grid = pd.read_csv(f"{OUT}/grid.csv"); front = pd.read_csv(f"{OUT}/front.csv")
print(len(grid), "designs evaluated,", int((grid["gates"].fillna("") == "").sum()), "pass the gates,", len(front), "on the exact front")
print(grid[["endurance_h", "mtow_kg"]].describe().loc[["min", "50%", "max"]])
plt.plot(front["mtow_kg"], front["endurance_h"], "o-", ms=3); plt.xlabel("take-off mass (kg)"); plt.ylabel("endurance (h)")
plt.title("exact Pareto front of the low-fidelity model"); plt.grid(alpha=.3); plt.show()

## What to report back
1. The cost line of section 3 and the `record.json` of section 4.
2. The three counts of section 5. Claude copies `grid.csv`, `front.csv` and `record.json` from Drive into
   `runs/drone_grid/<date>/` and commits them; nothing is retyped (the record carries a sha256 of each file).